# RMSNorm
*Root Mean Square Layer Normalization — the efficient normalization in LLaMA, Mistral, and Gemma.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_block_per_row.py` → `sm89_v2_smem_tree_reduce.py`


## What Is RMSNorm?

**In plain English:** RMSNorm rescales each token's vector so its "energy" (measured by the root-mean-square) equals 1, then multiplies by a learned per-dimension scale factor.

Think of it as **automatic volume control**: whatever the signal level coming in, normalize it to a consistent energy, then apply a learned gain.

**Why it exists:** Raw activations in deep networks tend to grow or shrink exponentially across layers — training becomes unstable. Normalization pins the magnitude so every layer receives well-scaled input.

**Where it appears:** Before every attention and FFN sub-layer in LLaMA, LLaMA 2/3, Mistral, Gemma, Qwen, GLM-4.


In [ ]:
import math
print('ready')

## Worked Example: Step by Step

**Input vector** (one token's representation):
$$x = [3.0,\ 4.0,\ 0.0]$$

**Learned scale:**
$$\gamma = [1.0,\ 1.0,\ 1.0] \quad \text{(identity — no scaling yet)}$$

| Step | Operation | Result |
|------|-----------|--------|
| 1 | $x^2 = [3^2,\ 4^2,\ 0^2]$ | $[9.0,\ 16.0,\ 0.0]$ |
| 2 | mean of $x^2 = (9+16+0)/3$ | $8.333$ |
| 3 | $\text{RMS} = \sqrt{8.333 + \varepsilon}$ | $2.887$ |
| 4 | $\text{inv\_rms} = 1 / 2.887$ | $0.346$ |
| 5 | output $= x \cdot \text{inv\_rms} \cdot \gamma$ | $[1.039,\ 1.386,\ 0.000]$ |

**Verification:** $\text{RMS}(\text{output}) = \sqrt{(1.039^2 + 1.386^2 + 0^2)/3} = \sqrt{1.0} = 1.0$ ✅


In [ ]:
x = [3.0, 4.0, 0.0]
gamma = [1.0, 1.0, 1.0]
eps = 1e-5
d = len(x)

sq = [xi**2 for xi in x]
mean_sq = sum(sq) / d
rms = math.sqrt(mean_sq + eps)
inv_rms = 1.0 / rms
output = [xi * inv_rms * gi for xi, gi in zip(x, gamma)]

print(f"x²         = {sq}")
print(f"mean(x²)   = {mean_sq:.4f}")
print(f"RMS        = {rms:.4f}")
print(f"inv_rms    = {inv_rms:.4f}")
print(f"output     = {[round(v, 4) for v in output]}")

# Verify: RMS of output should be ≈ 1.0
rms_out = math.sqrt(sum(v**2 for v in output) / d)
print(f"\nRMS(output) = {rms_out:.6f}  (should be 1.0 ✓)")
assert abs(rms_out - 1.0) < 1e-5


## The Formula

$$\boxed{\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i}$$

where

$$\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \varepsilon}$$

### Symbol Definitions

| Symbol | Name | Value in our example |
|--------|------|----------------------|
| $x_i$ | input activation at index $i$ | $[3.0, 4.0, 0.0]$ |
| $d$ | dimension of the vector | $3$ |
| $\sum_{i=1}^{d} x_i^2$ | sum of squares | $9 + 16 + 0 = 25$ |
| $\frac{1}{d}(\cdots)$ | mean of squares | $25/3 = 8.333$ |
| $\sqrt{\cdots}$ | square root | $2.887$ |
| $\varepsilon$ | small constant (epsilon) | $10^{-5}$, prevents division by zero |
| $\gamma_i$ | learned scale parameter | Initialized to 1, learned during training |

### 🗣️ Say It Out Loud
> *"RMSNorm of x at index i equals x-sub-i divided by the RMS of x, times gamma-sub-i. The RMS is the square root of the mean of the squares, plus a tiny epsilon for safety."*

**Tutor's gloss:** "We're measuring the 'energy' of the vector with $\frac{1}{d}\sum x_i^2$ — that's just the average of the squared values. Taking the square root gives us the root-mean-square. Dividing each element by this normalizes the total energy to 1. Then $\gamma$ lets the model re-scale however it learned to prefer."


## RMSNorm vs LayerNorm

LayerNorm (Ba et al. 2016) subtracts the mean first, then divides by the std:

$$\text{LN}(x)_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} \cdot \gamma_i + \beta_i$$

where $\mu = \frac{1}{d}\sum x_i$ and $\sigma^2 = \frac{1}{d}\sum (x_i - \mu)^2$.

**RMSNorm drops both $\mu$ subtraction and $\beta$.**

### Why? (Zhang & Sennrich, 2019)

The re-centering step (subtract $\mu$) is redundant when the network has bias terms elsewhere. The scale $\gamma$ is sufficient without the shift $\beta$.

**Result:** RMSNorm matches LayerNorm quality at lower cost:
- **One fewer pass** over the data (no mean scan needed)
- **7–64% faster** depending on model size

### Same vs Different Example


In [ ]:
def rmsnorm(x, gamma, eps=1e-5):
    d = len(x)
    rms = math.sqrt(sum(xi**2 for xi in x) / d + eps)
    return [xi / rms * gi for xi, gi in zip(x, gamma)]

def layernorm(x, gamma, beta, eps=1e-5):
    d = len(x)
    mu = sum(x) / d
    var = sum((xi - mu)**2 for xi in x) / d
    std = math.sqrt(var + eps)
    return [(xi - mu) / std * gi + bi for xi, gi, bi in zip(x, gamma, beta)]

gamma = [1.0] * 3
beta  = [0.0] * 3

# Zero-mean input: LN and RMSNorm give same result
x_zero_mean = [1.0, -1.0, 0.0]
rms_out = rmsnorm(x_zero_mean, gamma)
ln_out  = layernorm(x_zero_mean, gamma, beta)
print(f"x = {x_zero_mean}  (zero mean)")
print(f"  RMSNorm: {[round(v,4) for v in rms_out]}")
print(f"  LayerNorm: {[round(v,4) for v in ln_out]}")
print(f"  Same? {all(abs(a-b)<1e-5 for a,b in zip(rms_out, ln_out))}\n")

# Non-zero-mean input: they DIFFER
x_shifted = [3.0, 4.0, 0.0]
rms_out2 = rmsnorm(x_shifted, gamma)
ln_out2  = layernorm(x_shifted, gamma, beta)
print(f"x = {x_shifted}  (mean = {sum(x_shifted)/3:.2f}, non-zero)")
print(f"  RMSNorm: {[round(v,4) for v in rms_out2]}")
print(f"  LayerNorm: {[round(v,4) for v in ln_out2]}")
print(f"  Same? {all(abs(a-b)<1e-5 for a,b in zip(rms_out2, ln_out2))}")
print("  → They differ because LN subtracts the mean first")


## Optimization Ladder

| Version | Threads | Passes | Key idea |
|---------|---------|--------|----------|
| `v0_naive` | 1 | 2 | One thread: sum-of-squares, then normalize |
| `sm89_v1_block_per_row` | 128 | 2 | Parallel partial sums, tree reduce in SMEM |
| `sm89_v2_smem_tree_reduce` | 128 | 2 | Vectorized loads (`float4`), better SMEM layout |

**Why 2 passes?**
RMSNorm needs the full RMS before it can normalize any element.
Pass 1: compute $\sum x_i^2$ → get RMS.
Pass 2: apply $x_i / \text{RMS} \cdot \gamma_i$.
(Online single-pass isn't possible here unlike softmax.)
